In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models
import os

print("TensorFlow version:", tf.__version__)

# -------------------------------------
# 1. Load sample dataset (CIFAR-10)
# -------------------------------------
(x_train, y_train), (x_val, y_val) = tf.keras.datasets.cifar10.load_data()

# Normalize images
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0

# Resize images to 224x224 (needed for ResNet/MobileNet)
resize_layer = tf.keras.Sequential([
    layers.Resizing(224, 224)
])

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .batch(32)
    .map(lambda x, y: (resize_layer(x), y))
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(32)
    .map(lambda x, y: (resize_layer(x), y))
    .prefetch(tf.data.AUTOTUNE)
)

# -------------------------------------
# 2. Transfer Learning with ResNet50
# -------------------------------------
base_resnet = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_resnet.trainable = False  # freeze backbone

resnet_model = models.Sequential([
    base_resnet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax")  # CIFAR-10 has 10 classes
])

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n Training ResNet50 (frozen base)...")
history_1 = resnet_model.fit(train_ds, validation_data=val_ds, epochs=3)

# -------------------------------------
# 3. Fine-tuning (unfreeze last 30 layers)
# -------------------------------------
base_resnet.trainable = True
for layer in base_resnet.layers[:-30]:
    layer.trainable = False

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # lower LR for fine-tuning
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n Fine-tuning ResNet50 (last layers)...")
history_2 = resnet_model.fit(train_ds, validation_data=val_ds, epochs=3)

# -------------------------------------
# 4.  Save and Load Model (Keras format)
# -------------------------------------
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "resnet50_cifar10.keras")

# Save model
resnet_model.save(model_path)
print(f"\n Model saved successfully at: {model_path}")

# Load model to verify
loaded_model = tf.keras.models.load_model(model_path)
loss, acc = loaded_model.evaluate(val_ds)
print(f"\n Loaded model accuracy: {acc:.4f}")


TensorFlow version: 2.19.0

 Training ResNet50 (frozen base)...
Epoch 1/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 177s 106ms/step - accuracy: 0.1082 - loss: 2.3617 - val_accuracy: 0.1046 - val_loss: 2.2951
Epoch 2/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 154s 99ms/step - accuracy: 0.1219 - loss: 2.2925 - val_accuracy: 0.1903 - val_loss: 2.2805
Epoch 3/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 155s 99ms/step - accuracy: 0.1419 - loss: 2.2755 - val_accuracy: 0.1756 - val_loss: 2.2556

 Fine-tuning ResNet50 (last layers)...
Epoch 1/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 241s 142ms/step - accuracy: 0.3162 - loss: 1.8895 - val_accuracy: 0.4791 - val_loss: 1.4523
Epoch 2/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 241s 134ms/step - accuracy: 0.4907 - loss: 1.4288 - val_accuracy: 0.4861 - val_loss: 1.4250
Epoch 3/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 209s 134ms/step - accuracy: 0.5317 - loss: 1.3217 - val_accuracy: 0.4993 - val_loss: 1.3771

 Model saved successfully at: saved_models/resnet50_cifar10.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 34s 

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# -------------------------------------
# Load and preprocess the image
# -------------------------------------
img_path = "/content/cat.jpg"   # 🔹 change path as needed
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)  # batch dimension

# -------------------------------------
# Make prediction
# -------------------------------------
pred_probs = resnet_model.predict(img_array)
pred_class = np.argmax(pred_probs, axis=1)[0]
pred_label = class_names[pred_class]

print(f"✅ Predicted class: {pred_label} ({pred_probs[0][pred_class]*100:.2f}% confidence)")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
✅ Predicted class: bird (27.88% confidence)


In [ ]:
from google.colab import drive
import os


# 2️⃣ Create a folder path inside Drive to store model
save_dir = "/content/drive/MyDrive/ResNet_Model"
os.makedirs(save_dir, exist_ok=True)

# 3️⃣ Define the model path (.keras format)
model_path = os.path.join(save_dir, "resnet50_cifar10.keras")

# 4️⃣ Save model directly to Drive
# The model was already saved to /content/resnet50_cifar10.keras,
# so we can just copy it to the desired location in Drive.
# Alternatively, if the model object 'resnet_model' is still available
# from previous cells, you can save it again directly.

# Option 1: If resnet_model object is available (from cell nejyxN0xMNck)
if 'resnet_model' in locals():
  resnet_model.save(model_path)
  print(f"\n✅ Model saved successfully in Google Drive at:\n{model_path}")
else:
  # Option 2: If resnet_model is not available, copy from /content/
  source_path = "/content/resnet50_cifar10.keras"
  if os.path.exists(source_path):
    import shutil
    shutil.copy(source_path, model_path)
    print(f"\n✅ Model copied successfully to Google Drive at:\n{model_path}")
  else:
    print(f"\n ❌ Error: Model file not found at {source_path}")


✅ Model copied successfully to Google Drive at:
/content/drive/MyDrive/ResNet_Model/resnet50_cifar10.keras
